# Lesson 31: Electrochemical Impedance Spectroscopy

With improvements in computers and electronic engineering, the complexity of measurement techniques in electrochemistry has continually expanded. One particularly powerful approach involves the oscillation of voltages at a designated frequency, and analysis of the resulting current flow.

Today's topics:
- Electrochemical Impedance Spectroscopy
- Ideal behavior: resistors and capacitors
- Application to electrochemical systems

**Reference:**
- Ch.10 in *Electrochemical Methods: Fundamentals and Applications*, Allen J. Bard & Larry R. Faulkner.
---

## Configure Environment

In [ ]:
import sys
if '../Resources' not in sys.path:
    sys.path.append('../Resources')     # so that echem.py can be found

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from IPython.display import display
import schemdraw
import schemdraw.elements as elm

import echem                            # the shared toolkit for this course
from echem import R, T, F, f, WIDE      # constants and the two-panel figure size
echem.use_style()                       # course-wide graph formatting

# 1. Electrochemical Impedance Spectroscopy

The term *spectroscopy* means that we are now working with waves and frequencies. Electrochemical impedance spectroscopy (EIS) is the entry level to sinusoidal waveform-based measurements, with significantly more complex experiments possible. For example, alternating-current voltammetry, harmonic alternating-current voltammetry and intermodulation voltammetry are all possible. These techniques can be immensely powerful for analysis, but the results are incredibly complex and must be approached with care - even by experts in the field. This course will focus exclusively on EIS.

## EIS Waveform

The waveform is a sinusoidal voltage with time. The basic parameters to control the waveform include:
- $\omega$ : the frequency of voltage oscillations. This value can vary over ~10 orders of magnitude
- $E_{amp}$ : the amplitude of oscillations. This is typically chosen to be small (~5-10 mV) such that the system remains near equilibrium during the measurement.
- $E_{eq}$ : the potential at which of the electrochemical system is equilibrated during measurement. 


In [ ]:
@interact(frequency=widgets.FloatLogSlider(value=100, min=-3, max=6,
                                           description=r'$f$ (Hz)'),
          amp=widgets.FloatText(value=0.005, step=0.001, description=r'$E_{amp}$ (V)'),
          cycles=widgets.BoundedIntText(value=1, min=1, max=10, description='Repetitions'))
def plot_eis_waveform(frequency, amp, cycles):
    '''The EIS perturbation: a small sine wave about the resting potential.

    Note that this slider sets the ORDINARY frequency f, in Hz.  The angular
    frequency in the impedance equations is omega = 2 pi f.
    '''
    t, e = echem.sine_waveform(frequency, amplitude=amp, cycles=cycles)

    fig = plt.figure()
    ax = plt.subplot()
    ax.set_xlabel(r'$t$ /s')
    ax.set_ylabel(r'$E$ /V vs $E_{eq}$')
    ax.plot(t, e)
    ax.set_title(rf'$\omega = 2\pi f$ = {2*np.pi*frequency:.1f} rad s$^{{-1}}$', fontsize=8)

## Mathematical Models

From previous discussion, it should be apparent that this perturbation to voltage will result in a perturbation in current that must be monitored. To develop a mathematical model for this we treat both voltage and current as rotating vectors, referred to as *phasors*:

<img src='../Resources/images/phasor.png' width=30%>

These phasors are described by:
1. **Magnitude**: the maximum amplitude of the vector:

 $ \large \dot{E} = E_{amp}$ 
 
2. **Direction**: the direction in which the vector points, given here as the angle $\theta$ 
3. **Rotation rate**: the oscillation frequency $\omega$ :

From a conceptual basis, one aligns their eye along the $\theta=0$ axis and monitors the tip of the arrow representing the phasor. All that one would see is an up-down oscillation, which is exactly the sine-wave present in the interactive figure above. Mathematically, we represent this:

$
\large e = \dot{E}sin(\omega t)
$

Where the $e$ variable is deflection of the arrow tip from the $\theta=0$ axis.

Current and voltage are related by Ohm's law, which means that a sinusoidal response in current flow through the system should occur with at an identical frequency. We are not certain, however, whether these oscillations will be synchronized, or at what magnitude they will be. A similar treatment can be done for the current, with the introduction of a **phase shift** ( $\phi$ ). This parameter signifies the phase angle difference between the voltage and current phasors:

$
\large i = \dot{I}sin(\omega t + \phi)
$

Since both vectors rotate at the same frequency, we can experimentally use one phasor (most commonly $e$ , which is more easily controlled) as a reference point for the second phasor (typically $i$ ).

The experiment therefore involves the sinusoidal oscillation of $e$ at chosen $\omega$ for time $t$ while simultaneously measuring the response in $i$ . The response in $i$ tells us $\dot{I}$ , and comparison with $e$ yields $\phi$ . This complete characterization then enables interpretation of exactly what is occurring in the chemical system. In the most common form of experiment, EIS is performed at varied $\omega$ values such that the frequency-dependent response of the system can be monitored. This provides great insight into the nature of the reactions, and enables analysis of reaction kinetics across a very wide range of timeframes. More advanced analysis explores voltage-dependent and frequency-dependent changes. The mathematical framework must be developed to handle a response to the sinusoidal perturbation.

In [ ]:
@interact(frequency=widgets.FloatLogSlider(value=100, min=-3, max=6,
                                           description=r'$f$ (Hz)'),
          amp=widgets.FloatText(value=0.005, step=0.001, description=r'$E_{amp}$ (V)'),
          cycles=widgets.BoundedIntText(value=1, min=1, max=10, description='Repetitions'),
          phi=widgets.FloatSlider(value=45, min=0, max=360, step=5,
                                  description='Phase Shift (degrees)'),
          iamp=widgets.FloatText(value=0.005, step=0.001, description=r'$i_{amp}$ (A)'))
def plot_phase_shift(frequency, amp, phi, iamp, cycles):
    '''Voltage and current oscillate at the same frequency, but not in step.

    The phase angle phi between them is the measurement.  phi = 0 is a pure
    resistor; phi = 90 degrees is a pure capacitor.
    '''
    t, e = echem.sine_waveform(frequency, amplitude=amp, cycles=cycles)
    _, i = echem.sine_waveform(frequency, amplitude=iamp, cycles=cycles, phase_deg=phi)

    fig = plt.figure()
    ax = plt.subplot()
    ax2 = plt.twinx(ax)
    ax.set_xlabel(r'$t$ /s')
    ax.set_ylabel(r'$E$ /V vs $E_{eq}$')
    ax2.set_ylabel(r'$i$ /A')

    ax.plot(t, e, 'blue', label=r'$E$')
    ax2.plot(t, i, 'red', label=r'$i$')
    fig.legend(fontsize=6, loc=1)

# 2. Electronic Components

Analysis of EIS results requires that we simulate (or fit) the obtained results using theoretical circuits. A basic understanding of circuit elements is therefore necessary.

## Resistors

A resistor is simply a body that resists the flow of electrons according to Ohm's law. From the perspective of EIS, our experimental variables are $e$ and $i$ :

$
\large e = iR
$

The symbol for a resistor is:

In [ ]:
two_electrode_circuit = schemdraw.Drawing(inches_per_unit=0.75)
two_electrode_circuit.add(elm.Resistor(label='Resistor (R)', fontsize=12))

# annotate diagram
fig = two_electrode_circuit.draw()
two_electrode_circuit.save('../Workspace/R.png')
display(fig)

Based on the equations for our phasors above, we can also write out variations on Ohm's law using the amplitude of the vectors:

$
\large \dot{E} = \dot{I}R
$

and:

$
\large \dot{I} = \dot{E}/R
$

From these equations, it is clear that there is **no angle dependence**. Therefore, the two vectors can be considered to be aligned. This means that they will be **in-phase** and an EIS measurement will show perfect synchronization (note that the units are different and overlap of magnitudes is not meaningful):

In [ ]:
# A pure resistor: current follows voltage exactly.  Phase angle is zero.
FREQUENCY = 1.0     # Hz
CYCLES = 1
AMP_E = 0.005       # V
AMP_I = 0.005       # A

t, e = echem.sine_waveform(FREQUENCY, amplitude=AMP_E, cycles=CYCLES)
_, i = echem.sine_waveform(FREQUENCY, amplitude=AMP_I, cycles=CYCLES, phase_deg=0)

fig = plt.figure()
ax = plt.subplot()
ax2 = plt.twinx(ax)
ax.set_xlabel(r'$t$ /s')
ax.set_ylabel(r'$E$ /V vs $E_{eq}$')
ax2.set_ylabel(r'$i$ /A')
ax.plot(t, e, 'blue', label=r'$E$')
ax2.plot(t, i, 'red', label=r'$i$', linestyle='--')
ax.set_title(r'Resistor: $\phi = 0$', fontsize=8)
fig.legend(fontsize=6)

fig.savefig('../Workspace/EIS_resistor.png')

## Capacitors

A capacitor is a component that can store electrical charge across a voltage. The simplest form of capacitor is a parallel plate capacitor, where two metal plates are held parallel and a voltage is applied: it temporarily passes current to accumulate charge on the surfaces.

The symbol for a capacitor is:

In [ ]:
two_electrode_circuit = schemdraw.Drawing(inches_per_unit=0.75)
two_electrode_circuit.add(elm.Capacitor(label='Capacitor (C)', fontsize=12))

# annotate diagram
fig = two_electrode_circuit.draw()
two_electrode_circuit.save('../Workspace/C.png')
display(fig)

Mathematically, the charge ( $q$ ) stored by a capacitor is determined by its **capacitance** (C) and voltage ( $e$ ):

$
\large q = Ce
$

Charge is defined as the integral of current over time ( $q = it$ ). Therefore, we can rewrite:

$
\large i = C\frac{\delta e}{\delta t}
$

Recall that:

$
\large e = \dot{E}sin(\omega t)
$

Taking the first derivative of this equation with respect to time:

$
\large \frac{\delta e}{\delta t} = \omega\dot{E}cos(\omega t)
$

Substituting these equations:

$
\large i = \omega \dot{E} C cos(\omega t)
$

A cosine wave is simply a sine wave that is shifted by 90 degrees (or $\pi$ /2 radians):

$
\large i = \omega \dot{E} C sin(\omega t + \pi/2)
$

It is common to alter this by defining a parameter $X_C$ :

$
\large X_C = \frac{1}{\omega C}
$

Such that:

$
\large i = \frac{\dot{E}}{X_C} sin(\omega t + \pi/2)
$

The key outcome here is that both $e$ and $i$ change during the experiment and both have an angle dependence. The phasors are $\pi/2$ out of phase such that $\dot{E}$ is real and $\dot{I}$ is imaginary. Mathematically:

$
\large \dot{E} = -j X_C \dot{I}
$

The $\phi = \pi/2$ results in a shifted response in measured current:

In [ ]:
# A pure capacitor: current LEADS voltage by 90 degrees.
_, i_capacitor = echem.sine_waveform(FREQUENCY, amplitude=AMP_I,
                                     cycles=CYCLES, phase_deg=90)

fig = plt.figure()
ax = plt.subplot()
ax2 = plt.twinx(ax)
ax.set_xlabel(r'$t$ /s')
ax.set_ylabel(r'$E$ /V vs $E_{eq}$')
ax2.set_ylabel(r'$i$ /A')
ax.plot(t, e, 'blue', label=r'$E$')
ax2.plot(t, i_capacitor, 'red', label=r'$i$', linestyle='--')
ax.set_title(r'Capacitor: $\phi = \pi/2$', fontsize=8)
fig.legend(fontsize=6)

fig.savefig('../Workspace/EIS_capacitor.png')

## Summary

These two circuit elements are by far the most common used - in fact, you can likely do an entire lifetime of research in electrochemistry and rarely use more than these two circuit elements. The most important difference between them is the phase angle, $\phi$ . To summarize expectations:

- $\phi=0$ is expected for a pure resistor
- $\phi=\pi/2$ is expected for a pure capacitor
- $0 \lt \phi < \pi/2$ will be observed for anything in between 

<div class="alert alert-block alert-info">
<b>Activity:</b> Without looking below, predict the phase angle you would measure for each of these, and say which circuit element each one behaves like.
</div>

| System | Predicted $\phi$ | Behaves like |
|---|---|---|
| Electrolyte solution between two electrodes | ANSWER | ANSWER |
| A perfectly polarizable electrode passing no Faradaic current | ANSWER | ANSWER |
| An electrode with fast electron transfer at high frequency | ANSWER | ANSWER |

There are many other circuit elements that can be utilized, provided there is a rational reason to do so. One common element that you may come across in your literature searches is the **constant phase element**, which represents an imperfect capacitor. By this, we mean that the measured capacitance values are not independent of frequency. This has the effect of distorting the data relative to what is expected. The symbol for this element is:

In [ ]:
two_electrode_circuit = schemdraw.Drawing(inches_per_unit=0.75)
two_electrode_circuit.add(elm.CPE(label='Constant phase element (Q)', fontsize=12))

# annotate diagram
fig = two_electrode_circuit.draw()
two_electrode_circuit.save('../Workspace/Q.png')
display(fig)

# 3. Combining Circuit Elements

## RC Series
The combination of circuit elements together will alter their behavior. Consider for example the placement of a resistor and a capacitor **in series**:

In [ ]:
two_electrode_circuit = schemdraw.Drawing(inches_per_unit=0.75)
two_electrode_circuit.add(elm.Resistor())
two_electrode_circuit.add(elm.Capacitor())

# annotate diagram
fig = two_electrode_circuit.draw()
two_electrode_circuit.save('../Workspace/RC_series.png')
display(fig)

When placed in series, the voltage across the cell would be additive:

$
\large \dot{E} = \dot{E_R} + \dot{E_C}
$

The relationship developed between $\dot{E}$ and $\dot{I}$ for the capacitor can be substituted in to yield:

$
\large \dot{E} = \dot{I}(R-jX_C)
$

Which we will redefine as:

$
\large \dot{E} = \dot{I}Z
$

Where $Z$ is **impedance** of the cell, which is frequently represented using numerous conventions:

$
\large Z = R-jX_C = Z_{re} - jZ_{im} = Z' - jZ''
$

If we bring back the original definition of $X_C$ , we see the frequency dependence of the circuit:

$
\large Z(\omega) = R - \frac{j}{\omega C}
$

Again, the EIS experiment is to oscillate $e$ at varied $\omega$ and monitor $i$ and $\phi$ . The results can be presented in numerous ways, with each having a particular strength. The two most common are the **Nyquist representation** and the **Bode representation**.

The Nyquist representation plots $Z_{re}$ (real resistance) on the X-axis and $Z_{im}$ on the Y-axis. For this particular circuit, the resistor offers a static $Z_{re}$ , while the capacitor induces a change in $Z_{im}$ with frequency:

<img src='../Resources/images/nyquist.png' width=30%>

The Bode representation plots the total impedance $Z$ and the phase angle $\phi$ as a function of frequency:

<img src='../Resources/images/bode.png' width=60%>

Our earlier discussion established that $\phi$ values of 0 and 90 signify pure resistance and pure capacitance, respectively. Therefore, the Bode representation provides us valuable insights in this situation which the Nyquist representation doesn't: the placement of a resistor and capacitor in series results in a transition from one extreme to another. At high frequencies of $e$ variation we will see only the resistor, but at low frequencies we will see only the capacitor. 

## RC Parallel

Let us consider the other simple case, the placement of a resistor and capacitor in parallel:

In [ ]:
two_electrode_circuit = schemdraw.Drawing(inches_per_unit=0.75)
two_electrode_circuit.add(elm.Dot(open=True).right())
two_electrode_circuit.add(elm.Line().right())
two_electrode_circuit.add(elm.Resistor(label=r'R = 100 $\Omega$'))
two_electrode_circuit.add(elm.Line().down())
two_electrode_circuit.add(elm.Line().right())
two_electrode_circuit.add(elm.Dot(open=True).left())

two_electrode_circuit.add(elm.Line().left())
two_electrode_circuit.add(elm.Capacitor(label=r'C = 1 $\mu F$').left())
two_electrode_circuit.add(elm.Line().up())

# annotate diagram
fig = two_electrode_circuit.draw()
two_electrode_circuit.save('../Workspace/RC_parallel.png')
display(fig)

In this situation, the Nyquist representation provides much useful information (numbers on graph represent $\omega$ at the denoted points):

<img src='../Resources/images/nyquist-parallel.png' width=30%>

Careful inspection of this semicircular graph should reveal that many of our parameters can be rapidly extracted:

1. **Resistance**: the diameter of the semicircle is exactly equal to the resistance of the resistor element
2. **Capacitance**: the apex of the circle corresponds to the capacitance of the capacitor element. Specifically:

$ 
\large \omega = \frac{1}{RC}
$

As will be seen, this observation of semicircular data is critically important for electrochemistry. The ability to quickly extract meaningful information at a glance is a characteristic of Nyquist plots.

The Bode representation:

<img src='../Resources/images/bode-parallel.png' width=30%>

We can once again see the transition between purely capacitive and purely resistive behavior, which is reversed compared to the elements placed in series with each other. This is quickly deduced by inspection of the phase angle portion of the data. Inspection of the data rapidly yields the:

1. **Resistance**: the height of the impedance plateau where $\phi=0$ 

There is no direct way to extract capacitance from this data, it would require mathematical simulation.

# 4. Application to Electrochemistry

Application of EIS to electrochemistry relies on the development of a circuit model that emulates the observed experimental behavior. The process often involves an initial guess, followed by a least-squares fitting routine. As is the case with fitting of data, this process is full of pitfalls!

Some Warnings: 
1. **No circuit is unique.** You can always draw another circuit that will yield the same behavior.
2. **Frequency dispersion** is the variation of results with $\omega$ . This can occur even in our "ideal" parameters ( $R_\omega$ and $C$ ).
3. **Statistical analysis** is a requirement for any modelling. This necessitates careful thought and justification of your model.

The basic mathematical rules in building circuits:

**Resistors in series** are additive:

$
\large R_T = R_1 + R_2 + ... R_n
$

**Resistors in parallel** are additive in reciprocal space:

$
\large \frac{1}{R_T} = \frac{1}{R_1} + \frac{1}{R_2} + ... \frac{1}{R_n}
$

**Impedance** ( $Z$ ) is a generalized form of resistance, we rely on application of these rules to simulate the model. This is rarely done by hand - software packages that enable you to "draw" circuits and fit experimental results are readily available.

## The Randles Circuit

The Randles-circuit is ubiquitous in electrochemistry. It is the basic component used to describe electrochemical cells. The circuit, marked up with the charging current ( $i_c$ ) and Faradaic current ( $i_f$ ) is:

<img src='../Resources/images/randles.png' width=60%>

The components are:

- $R_\Omega$ : resistance in the electrolyte solution of the electrochemical cell and instrumentation
- $C_d$ : double layer capacitance
- $Z_f$ : a complex circuit element consisting of a resistive and capacitive element. Multiple forms exist, but most commonly they are a series RC or a resistor with a *Warburg element*
- $R_{ct}$ : charge transfer resistance (see [Lesson 12](L12-Kinetics-II-Tafel.ipynb))
- $Z_w$ : Warburg element; captures mass transfer limitations in the electrochemical response

The simulated Nyquist response of this circuit showing the effect of a Warburg element:

<img src='../Resources/images/randles-nyquist.png' width=60%>

## Useful Applications

EIS is capable of directly measuring $i_0$ and $k^0$ values. If the system is analyzed in the particular situation where $C_O(0,t) = C_R(0,t)$ , then:

$
\large R_{CT} = \frac{RT}{F i_0} = R_s - \frac{1}{\omega C_s}
$

Where $R_s$ and $C_s$ are the resistance and capacitance of the **series equivalent circuit** measured at frequency $\omega$ -- not the solution resistance $R_\Omega$ (see Bard & Faulkner eq. 10.3.5).

The limitations to such measurements are dependent on the **electrochemical time constant** of the cell, $\tau$ :

$
\large \tau = R_\Omega C_d
$

The **upper limit** in capabilities of EIS is when 

$
\large \tau < 1/\omega
$ 

At this point:

$
\large k^0 \leq \sqrt{\frac{D \omega}{2}}
$

For typical systems, this represents $k^0 \leq$ 10 cm s $^{-1}$ 

The **lower limit** is near:

$
\large R_{CT} \leq \frac{1}{\omega C_d}
$

For typical systems, this represents $k^0 \geq 10^{-6} cm s^{-1}$ .

A simple EIS experiment, which takes several minutes, therefore enables direct measurement of $k^0$ over ~7 orders of magnitude.

---
# Literature Examples

EIS factors heavily into the analysis of semiconductors in all types of photovoltaics and photoelectrochemical technologies. For example:

[2012 J. Am. Chem. Soc. 134, 4294-4302](https://doi.org/10.1021/ja210755h)


<img src='../Resources/images/jacs2012.png' width=30%>

An extremely diverse range of electrochemical systems and behaviors yields an equally diverse range of behaviors to be studied. For example:

[2012 J. Phys. Chem. Lett. 3, 2517-2522.](https://doi.org/10.1021/jz3010909)

<img src='../Resources/images/2012jpcl.png' width=60%>

<div class="alert alert-block alert-info">
<b>Activity:</b> The text above warns that fitting a circuit model is "full of pitfalls" because more than one circuit can fit the same data.
<br>
Given a Nyquist plot that fits both a Randles circuit and a more elaborate alternative, list three pieces of evidence outside the fit itself that you would use to choose between them.
</div>

1. ANSWER
2. ANSWER
3. ANSWER

<div class="alert alert-block alert-danger">
<b>Memory Check:</b> $R_{CT} = RT/(Fi_0)$ lets EIS measure the exchange current directly. Where did $i_0$ come from, and what does a <b>small</b> charge transfer resistance tell you about the reaction?
</div>